# EXAONE Deep 7.8B QLoRA 파인튜닝 (Runpod용)
그룹 F: EXAONE Deep 7.8B + QLoRA

**사전 준비:**
- `finetune_train.jsonl` 파일을 `/workspace/`에 업로드
- HuggingFace 토큰 준비

In [ ]:
# Step 1. 패키지 설치
!pip install -q --upgrade transformers
!pip install -q peft trl bitsandbytes accelerate datasets huggingface_hub

import transformers, peft, trl
print(f'transformers: {transformers.__version__}')
print(f'peft: {peft.__version__}')
print(f'trl: {trl.__version__}')
print('설치 완료!')

In [ ]:
# Step 2. HuggingFace 로그인
from huggingface_hub import login
login(token="hf_여기에토큰입력")  # ← 본인 HF 토큰으로 변경!
print('로그인 완료!')

In [ ]:
# Step 3. 데이터 로드
import json
from datasets import Dataset

data = []
with open('/workspace/finetune_train.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

print(f'총 {len(data)}개 로드 완료!')
print('샘플:')
for m in data[0]['messages']:
    print(f"  [{m['role']}]: {m['content'][:60]}")

In [ ]:
# Step 4. 모델 & 토크나이저 로드 (4bit 양자화)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'LGAI-EXAONE/EXAONE-Deep-7.8B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('토크나이저 로드 중...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('모델 로드 중... (5~10분 걸려요)')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
print(f'완료! GPU 메모리: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# Step 5. LoRA 설정
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 6. 데이터셋 토크나이징
MAX_LENGTH = 512

def format_and_tokenize(item):
    text = ''
    for msg in item['messages']:
        role, content = msg['role'], msg['content']
        if role == 'system':
            text += f'[|system|]{content}[|endofturn|]\n'
        elif role == 'user':
            text += f'[|user|]{content}[|endofturn|]\n'
        elif role == 'assistant':
            text += f'[|assistant|]{content}[|endofturn|]'
    
    result = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

dataset = Dataset.from_list(data)
dataset = dataset.map(format_and_tokenize, remove_columns=['messages', 'dataset'])
print(f'데이터셋 준비 완료: {len(dataset)}개')

In [ ]:
# Step 7. 학습 (Runpod에서 풀로 돌리기!)
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = '/workspace/exaone_adapter'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,             # Runpod에서는 3 epoch 풀로!
    per_device_train_batch_size=4,  # 24GB GPU면 4 가능
    gradient_accumulation_steps=4,  # 실질 배치: 16
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    report_to='none',
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print('학습 시작!')
trainer.train()
print('학습 완료!')

In [ ]:
# Step 8. 어댑터 저장 & HF 업로드
import os

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print('저장된 파일들:')
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f}: {size/1e6:.1f} MB')

# HF Hub 업로드
HF_REPO = 'your-hf-username/exaone-deep-7.8b-finance'  # ← 본인 HF 계정명으로 변경!
model.push_to_hub(HF_REPO, private=True)
tokenizer.push_to_hub(HF_REPO, private=True)
print(f'HF 업로드 완료: {HF_REPO}')

In [ ]:
# Step 9. 추론 테스트
def inference(system, user_msg, max_new_tokens=64):
    prompt  = f'[|system|]{system}[|endofturn|]\n'
    prompt += f'[|user|]{user_msg}[|endofturn|]\n'
    prompt += f'[|assistant|]'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

result = inference(
    system='You are a financial sentiment analyst. Classify the given financial text as exactly one of: positive, negative, neutral. Reply with only the label, nothing else.',
    user_msg='Text: The company reported record profits this quarter, exceeding analyst expectations.'
)
print(f'예측: {result}')
print(f'정답: positive')